In [1]:
# ============================================================
# ScamShield AI — Notebook 5: Ensemble Risk Scorer
# ============================================================
# Combines ALL module outputs into a final 0-100 risk score.
#
# ENSEMBLE STRATEGY:
# ─────────────────
# Not all inputs are always available.
# User might provide: text only, URL only, image only, or all.
# We use WEIGHTED AVERAGE based on what's available.
#
# Weights (from our domain knowledge):
#   DistilBERT text:  40%  (most accurate, context-aware)
#   XGBoost text:     25%  (good at patterns)
#   URL classifier:   25%  (structural features reliable)
#   LR text:          10%  (baseline, but still useful)
#
# RISK LEVEL MAPPING:
#   0-25:   LOW      → Likely safe
#   26-50:  MEDIUM   → Be cautious
#   51-75:  HIGH     → Very suspicious
#   76-100: CRITICAL → Almost certainly a scam
# ============================================================

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

print("=" * 55)
print("  ScamShield AI — Ensemble Risk Scorer")
print("=" * 55)

# ── Risk Level Thresholds ─────────────────────────────────
RISK_LEVELS = {
    'LOW':      (0,  25,  '#2ecc71', '✅'),
    'MEDIUM':   (26, 50,  '#f39c12', '⚠️'),
    'HIGH':     (51, 75,  '#e67e22', '🚨'),
    'CRITICAL': (76, 100, '#e74c3c', '🔴'),
}

# ── Model Weights ─────────────────────────────────────────
DEFAULT_WEIGHTS = {
    'distilbert':  0.40,
    'xgboost_text': 0.25,
    'url_model':   0.25,
    'lr_text':     0.10,
}


def get_risk_level(score):
    """
    Convert 0-100 score to risk level.
    
    Args:
        score: Float 0-100
        
    Returns:
        tuple: (level_name, color, emoji)
    """
    for level, (low, high, color, emoji) in RISK_LEVELS.items():
        if low <= score <= high:
            return level, color, emoji
    return 'CRITICAL', '#e74c3c', '🔴'


def calculate_ensemble_score(
    text_proba_lr=None,
    text_proba_xgb=None,
    text_proba_bert=None,
    url_proba=None,
    weights=None
):
    """
    Calculate weighted ensemble risk score.
    
    Args:
        text_proba_lr:   Logistic Regression scam probability (0-1)
        text_proba_xgb:  XGBoost scam probability (0-1)
        text_proba_bert: DistilBERT scam probability (0-1)
        url_proba:       URL classifier scam probability (0-1)
        weights:         Custom weights dict (optional)
    
    Returns:
        dict: {
            'risk_score': 0-100 score,
            'risk_level': LOW/MEDIUM/HIGH/CRITICAL,
            'confidence': how many models contributed,
            'breakdown': individual model contributions
        }
    """
    
    if weights is None:
        weights = DEFAULT_WEIGHTS
    
    # Collect available predictions
    contributions = {}
    total_weight = 0.0
    weighted_sum = 0.0
    
    if text_proba_bert is not None:
        w = weights.get('distilbert', 0.40)
        contributions['DistilBERT'] = {
            'probability': float(text_proba_bert),
            'weight': w,
            'contribution': float(text_proba_bert * w)
        }
        weighted_sum += text_proba_bert * w
        total_weight += w
    
    if text_proba_xgb is not None:
        w = weights.get('xgboost_text', 0.25)
        contributions['XGBoost Text'] = {
            'probability': float(text_proba_xgb),
            'weight': w,
            'contribution': float(text_proba_xgb * w)
        }
        weighted_sum += text_proba_xgb * w
        total_weight += w
    
    if url_proba is not None:
        w = weights.get('url_model', 0.25)
        contributions['URL Classifier'] = {
            'probability': float(url_proba),
            'weight': w,
            'contribution': float(url_proba * w)
        }
        weighted_sum += url_proba * w
        total_weight += w
    
    if text_proba_lr is not None:
        w = weights.get('lr_text', 0.10)
        contributions['Logistic Regression'] = {
            'probability': float(text_proba_lr),
            'weight': w,
            'contribution': float(text_proba_lr * w)
        }
        weighted_sum += text_proba_lr * w
        total_weight += w
    
    # Handle case: no predictions available
    if total_weight == 0:
        return {
            'risk_score': 0,
            'risk_level': 'LOW',
            'confidence': 'No models',
            'breakdown': {}
        }
    
    # Normalize by actual weight used (handles missing models)
    normalized_prob = weighted_sum / total_weight
    
    # Convert probability to 0-100 score
    risk_score = round(normalized_prob * 100, 1)
    
    # Get risk level
    risk_level, color, emoji = get_risk_level(risk_score)
    
    # Confidence based on how many models contributed
    n_models = len(contributions)
    if n_models >= 3:
        confidence = 'High'
    elif n_models == 2:
        confidence = 'Medium'
    else:
        confidence = 'Low'
    
    return {
        'risk_score':  risk_score,
        'risk_level':  risk_level,
        'risk_color':  color,
        'risk_emoji':  emoji,
        'confidence':  confidence,
        'n_models':    n_models,
        'breakdown':   contributions
    }


print("Ensemble scoring function ready!")
print()
print("Risk Level Thresholds:")
for level, (low, high, color, emoji) in RISK_LEVELS.items():
    print(f"  {emoji} {level:<10}: {low:>3} - {high:>3}")

  ScamShield AI — Ensemble Risk Scorer
Ensemble scoring function ready!

Risk Level Thresholds:
  ✅ LOW       :   0 -  25
  ⚠️ MEDIUM    :  26 -  50
  🚨 HIGH      :  51 -  75
  🔴 CRITICAL  :  76 - 100


In [ ]:
# ============================================================
# TEST ENSEMBLE SCORER WITH REAL EXAMPLES
# ============================================================

test_scenarios = [
    {
        'name': 'Clear Scam (All models agree)',
        'text_proba_lr':   0.95,
        'text_proba_xgb':  0.92,
        'text_proba_bert': 0.97,
        'url_proba':       0.89,
    },
    {
        'name': 'Legitimate Message',
        'text_proba_lr':   0.05,
        'text_proba_xgb':  0.03,
        'text_proba_bert': 0.02,
        'url_proba':       0.04,
    },
    {
        'name': 'Suspicious but not certain',
        'text_proba_lr':   0.65,
        'text_proba_xgb':  0.70,
        'text_proba_bert': 0.55,
        'url_proba':       None,   # No URL in message
    },
    {
        'name': 'Text only (no URL)',
        'text_proba_lr':   0.85,
        'text_proba_xgb':  0.80,
        'text_proba_bert': 0.88,
        'url_proba':       None,
    },
    {
        'name': 'URL only (no text)',
        'text_proba_lr':   None,
        'text_proba_xgb':  None,
        'text_proba_bert': None,
        'url_proba':       0.91,
    },
]

print("Testing Ensemble Scorer with multiple scenarios:")
print()

results_list = []

for scenario in test_scenarios:
    name = scenario.pop('name')
    result = calculate_ensemble_score(**scenario)
    
    print(f"  Scenario: {name}")
    print(f"  {result['risk_emoji']} Risk Score: {result['risk_score']}/100 "
          f"— {result['risk_level']}")
    print(f"  Confidence: {result['confidence']} ({result['n_models']} models)")
    print(f"  Breakdown:")
    for model, data in result['breakdown'].items():
        print(f"    {model:<22}: prob={data['probability']:.3f} "
              f"× weight={data['weight']:.2f} = {data['contribution']:.3f}")
    print()
    
    results_list.append({'scenario': name, **result})

# ── Visualization of Risk Gauge ───────────────────────────
# Create a gauge chart for one example
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ScamShield AI — Risk Score Visualization', 
             fontsize=14, fontweight='bold')

# Left: Risk level color bar
scores    = [r['risk_score'] for r in results_list]
scenarios = [r['scenario'] for r in results_list]
colors    = [r['risk_color'] for r in results_list]

axes[0].barh(range(len(scores)), scores, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_yticks(range(len(scenarios)))
axes[0].set_yticklabels([s[:35] for s in scenarios], fontsize=9)
axes[0].set_xlabel('Risk Score (0-100)')
axes[0].set_title('Ensemble Risk Scores by Scenario')
axes[0].axvline(x=25, color='gray', linestyle='--', alpha=0.5, label='LOW/MEDIUM')
axes[0].axvline(x=50, color='orange', linestyle='--', alpha=0.5, label='MEDIUM/HIGH')
axes[0].axvline(x=75, color='red', linestyle='--', alpha=0.5, label='HIGH/CRITICAL')

# Add score labels
for i, (score, color) in enumerate(zip(scores, colors)):
    axes[0].text(score + 1, i, f'{score}', va='center', fontweight='bold')

# Add risk zone backgrounds
axes[0].axvspan(0, 25, alpha=0.05, color='green')
axes[0].axvspan(25, 50, alpha=0.05, color='yellow')
axes[0].axvspan(50, 75, alpha=0.05, color='orange')
axes[0].axvspan(75, 100, alpha=0.05, color='red')
axes[0].set_xlim(0, 110)
axes[0].legend(fontsize=8)

# Right: Model contribution breakdown for scenario 1
result_0 = results_list[0]
if result_0['breakdown']:
    models    = list(result_0['breakdown'].keys())
    contribs  = [result_0['breakdown'][m]['contribution'] for m in models]
    
    axes[1].pie(contribs, labels=models, autopct='%1.1f%%',
               colors=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'])
    axes[1].set_title(f"Score Contribution Breakdown\n"
                      f"Scenario: {scenarios[0][:30]}\n"
                      f"Final Score: {result_0['risk_score']}/100")

plt.tight_layout()
plt.show()


# Save ensemble configuration
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
MODELS_DIR = os.path.join(PROJECT_ROOT, 'backend', 'saved_models')

ensemble_config = {
    'weights': DEFAULT_WEIGHTS,
    'risk_levels': {
        level: {'min': low, 'max': high, 'color': color}
        for level, (low, high, color, emoji) in RISK_LEVELS.items()
    },
    'version': '1.0'
}

with open(os.path.join(MODELS_DIR, 'ensemble_config.json'), 'w') as f:
    json.dump(ensemble_config, f, indent=2)

print(f"Ensemble configuration saved!")

Testing Ensemble Scorer with multiple scenarios:

  Scenario: Clear Scam (All models agree)
  🔴 Risk Score: 93.5/100 — CRITICAL
  Confidence: High (4 models)
  Breakdown:
    DistilBERT            : prob=0.970 × weight=0.40 = 0.388
    XGBoost Text          : prob=0.920 × weight=0.25 = 0.230
    URL Classifier        : prob=0.890 × weight=0.25 = 0.223
    Logistic Regression   : prob=0.950 × weight=0.10 = 0.095

  Scenario: Legitimate Message
  ✅ Risk Score: 3.1/100 — LOW
  Confidence: High (4 models)
  Breakdown:
    DistilBERT            : prob=0.020 × weight=0.40 = 0.008
    XGBoost Text          : prob=0.030 × weight=0.25 = 0.007
    URL Classifier        : prob=0.040 × weight=0.25 = 0.010
    Logistic Regression   : prob=0.050 × weight=0.10 = 0.005

  Scenario: Suspicious but not certain
  🚨 Risk Score: 61.3/100 — HIGH
  Confidence: High (3 models)
  Breakdown:
    DistilBERT            : prob=0.550 × weight=0.40 = 0.220
    XGBoost Text          : prob=0.700 × weight=0.25 = 0.175